# 01 — Autonomous ETL Agent: Project Overview

**Course**: Agentic AI / Transformative GenAI for Data Engineers  
**Capstone**: Autonomous ETL Agent  

---

This notebook provides an interactive tour of the Autonomous ETL Agent project:
- What problem it solves
- The multi-agent architecture (LangGraph)
- The technology stack and key design decisions
- A live demo of the pipeline via the CLI and REST API

> 💡 **Tip**: Run each cell sequentially. Cells marked `⚙️ Setup` must be run first.

## 1. The Problem

Traditional ETL development is slow and repetitive:

```
Data Engineer receives requirement → writes spec → codes pipeline → writes tests
→ creates PR → requests review → fixes issues → merges → configures Airflow
```

This process takes **days to weeks** for each pipeline. With hundreds of pipelines in a
data platform, the bottleneck is always human bandwidth.

**The Autonomous ETL Agent solves this** by replacing the human in this loop with AI.

## 2. The Solution: Multi-Agent Pipeline

```
User Story (YAML)
       │
       ▼
┌─────────────────────────────────────────────────────────────────┐
│                    LangGraph State Machine                       │
│                                                                   │
│  StoryParser ──► CodingAgent ──► TestAgent ──► PRAgent ──► DeployAgent │
│  (Claude)        (Claude)        (pytest)      (GitHub)    (S3+Airflow) │
└─────────────────────────────────────────────────────────────────┘
       │                    │                    │
       ▼                    ▼                    ▼
   S3 Artifact          GitHub PR          Airflow DAG Run
```

### Agent Responsibilities

| Agent | Input | Output | LLM? |
|-------|-------|--------|------|
| **StoryParser** | YAML user story | ETLSpec (structured) | ✅ Claude |
| **CodingAgent** | ETLSpec | PySpark code + README | ✅ Claude |
| **TestAgent** | ETLSpec + code | pytest test file + results | ✅ Claude + subprocess |
| **PRAgent** | Code + test results | GitHub Issue + PR URL | ✅ Claude (commit msg) |
| **DeployAgent** | PR URL + code | S3 artifact + Airflow run | ❌ boto3 + httpx |

## 3. ⚙️ Setup

In [ ]:
# Install the project in editable mode (if not already installed)
import subprocess
result = subprocess.run(['uv', 'sync'], capture_output=True, text=True, cwd='../..')
print(result.stdout or result.stderr or 'Dependencies already installed')

In [ ]:
import sys
sys.path.insert(0, '../../src')

# Verify imports
from etl_agent.core.models import RunStatus, ETLOperation
from etl_agent.core.state import GraphState
print('✅ Imports successful')

## 4. The User Story Format

Everything starts with a YAML user story. Let's load one of the included examples:

In [ ]:
import yaml
from pathlib import Path

story_path = Path('../../config/story_examples/rfm_analysis.yaml')
with open(story_path) as f:
    story = yaml.safe_load(f)

print(f"Story ID   : {story['id']}")
print(f"Title      : {story['title']}")
print(f"Source     : {story['source']['path']} ({story['source']['format']})")
print(f"Target     : {story['target']['path']} ({story['target']['format']})")
print(f"Steps      : {len(story.get('transformations', []))} transformation(s)")
print(f"Criteria   : {len(story.get('acceptance_criteria', []))} acceptance criterion/ia")
print(f"Tags       : {story.get('tags', [])}")

In [ ]:
# Show all 6 included story examples
story_dir = Path('../../config/story_examples')
for yaml_file in sorted(story_dir.glob('*.yaml')):
    with open(yaml_file) as f:
        s = yaml.safe_load(f)
    steps = len(s.get('transformations', []))
    print(f"  📄 {yaml_file.name:<30} → {s['title']} ({steps} steps)")

## 5. The LangGraph State Machine

In [ ]:
from etl_agent.core.models import RunStatus
from etl_agent.core.state import route_after_tests, route_after_pr

# Demonstrate routing logic
print("=== LangGraph Routing Demonstration ===")
print()

# Tests pass → go to PR agent
from etl_agent.core.models import TestResult
state_pass = {
    'test_results': TestResult(passed=True, num_passed=5, num_failed=0, coverage_pct=90.0),
    'retry_count': 0, 'max_retries': 2, 'awaiting_approval': False
}
print(f"Tests PASS  → route: {route_after_tests(state_pass)}")

# Tests fail, retry available → go back to coding
state_fail_retry = {
    'test_results': TestResult(passed=False, num_passed=2, num_failed=3, coverage_pct=60.0),
    'retry_count': 0, 'max_retries': 2, 'awaiting_approval': False
}
print(f"Tests FAIL  → route: {route_after_tests(state_fail_retry)}")

# Tests fail, max retries → failure
state_fail_max = {
    'test_results': TestResult(passed=False, num_passed=0, num_failed=5, coverage_pct=40.0),
    'retry_count': 2, 'max_retries': 2, 'awaiting_approval': False
}
print(f"Max retries → route: {route_after_tests(state_fail_max)}")

# PR created → go to deploy
state_pr_ok = {'github_pr_url': 'https://github.com/org/repo/pull/42'}
print(f"PR created  → route: {route_after_pr(state_pr_ok)}")

## 6. RunStatus Lifecycle

In [ ]:
from etl_agent.core.models import RunStatus

print("=== Pipeline Run Status Lifecycle ===")
print()

flow = [
    (RunStatus.PENDING,           "Run accepted by API"),
    (RunStatus.PARSING,           "StoryParser agent processing YAML"),
    (RunStatus.CODING,            "CodingAgent generating PySpark code"),
    (RunStatus.TESTING,           "TestAgent running pytest"),
    (RunStatus.AWAITING_APPROVAL, "Paused for human review (optional)"),
    (RunStatus.PR_CREATING,       "PRAgent creating GitHub Issue + PR"),
    (RunStatus.DEPLOYING,         "DeployAgent uploading to S3 + Airflow"),
    (RunStatus.DONE,              "Pipeline completed successfully ✅"),
]

for i, (status, description) in enumerate(flow):
    connector = ' └─► ' if i == len(flow) - 1 else ' ├─► '
    print(f"{connector}{status.value:<22} {description}")

print()
print(" └─► FAILED               Any unrecoverable error ❌")

## 7. Tech Stack Summary

In [ ]:
tech_stack = [
    ("LLM",              "Anthropic Claude Sonnet 4.6",      "claude-sonnet-4-20250514"),
    ("Agent Framework",  "LangGraph + LangChain",            "Stateful multi-agent graph"),
    ("ETL Engine",        "PySpark 3.5 + Delta Lake",         "CRUD on Delta tables"),
    ("API",              "FastAPI + Uvicorn",                 "Async, SSE, rate limiting"),
    ("Database",          "SQLAlchemy 2.0 async",             "Postgres / SQLite"),
    ("Storage",           "AWS S3 (LocalStack local dev)",    "boto3"),
    ("Orchestration",     "Apache Airflow 2.x",               "REST API trigger"),
    ("Git Automation",    "PyGitHub",                         "Issues, branches, PRs"),
    ("Package Manager",   "UV",                               "Blazing fast Python deps"),
    ("Logging",           "structlog",                        "JSON structured logs"),
    ("Infrastructure",    "Terraform",                        "EC2, ECR, S3, IAM"),
    ("CI/CD",             "GitHub Actions",                   "lint + test + deploy"),
]

print(f"{'Layer':<20} {'Technology':<35} {'Notes'}")
print('-' * 80)
for layer, tech, notes in tech_stack:
    print(f"{layer:<20} {tech:<35} {notes}")

## 8. Quick API Demo

Make sure the services are running: `make up`

In [ ]:
import requests
import os

API_BASE = 'http://localhost:8000/api/v1'
API_KEY = os.getenv('API_KEY', 'your-api-key-here')
headers = {'X-API-Key': API_KEY}

# 1. Health check
try:
    resp = requests.get(f'{API_BASE}/health', timeout=5)
    print('Health:', resp.json())
except Exception as e:
    print(f'⚠️  Services not running: {e}')
    print('   Run `make up` first to start all services')

In [ ]:
# 2. Submit a user story
rfm_story = open('../../config/story_examples/rfm_analysis.yaml').read()

try:
    resp = requests.post(
        f'{API_BASE}/stories',
        json={'story_yaml': rfm_story, 'deploy': False, 'dry_run': False},
        headers=headers,
        timeout=10,
    )
    if resp.ok:
        data = resp.json()
        run_id = data['run_id']
        print(f'✅ Run submitted: {run_id}')
    else:
        print(f'Error {resp.status_code}: {resp.text}')
except Exception as e:
    print(f'⚠️  Cannot connect: {e}')

## 9. Next Steps

Continue with the other notebooks:

- **`02_pyspark_pipeline_walkthrough.ipynb`** — Deep dive into the generated PySpark code, Jinja2 templates, and Delta Lake operations
- **`03_business_analytics_demo.ipynb`** — Run the 4 business analytics pipelines (RFM, Geo, Campaign, Intent) on real Amazon fixture data